In [ ]:
import pandas as pd
import numpy as np

# Define options
num_warehouses_options = [2, 3, 4, 5]
num_customers_options = [50, 100, 200, 400]
num_customers_options = [50]
capacity_distribution_options = ['uniform', 'uneven']

# Initialize an empty list to store dataframes
results_list = []

# Loop through parameter combinations
for num_warehouses in num_warehouses_options:
    for num_customers in num_customers_options:
        for capacity_distribution in capacity_distribution_options:
            # Construct the filename
            filename = f'results_old/full_tables/full_results_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv'
            read_df = pd.read_csv(filename)

            # export_results.ipynb appends to these CSVs (mode='a') on every run, so a
            # policy that got re-run (e.g. while debugging one policy at a time) has
            # multiple rows for the same config. Keep only the last one per policy - the
            # most recent run - since append order == chronological order.
            read_df = read_df.drop_duplicates(subset='Policy', keep='last').reset_index(drop=True)

            # Identify the perfect hindsight reward
            perfect_hindsight_mean_reward = read_df.loc[read_df['Policy'] == 'perfect_hindsight', 'MeanReward'].values[0]

            # Create a new dataframe
            new_df = pd.DataFrame()
            new_df['num_warehouses'] = [num_warehouses] * len(read_df)
            new_df['num_customers'] = [num_customers] * len(read_df)
            new_df['capacity_distribution'] = [capacity_distribution] * len(read_df)

            # Add policy columns
            new_df['policy'] = read_df['Policy']
            new_df['avg_reward'] = read_df['MeanReward']
            new_df['std_reward'] = read_df['StdReward']
            new_df['mean_time'] = read_df['MeanTime']
            new_df['std_time'] = read_df['StdTime']
            new_df['all_rewards'] = read_df['AllRewards'].apply(lambda x: eval(x) if isinstance(x, str) else x)
            new_df['all_times_n_vectors'] = read_df['AllTimes'].apply(lambda x: eval(x) if isinstance(x, str) else x)
            new_df['all_distance_ranks_n_vectors'] = read_df['AllFacilityProximity'].apply(lambda x: eval(x) if isinstance(x, str) else x)

            #Get All_Distance_ranks single vector
            new_df['all_distance_ranks_single_vector'] = new_df['all_distance_ranks_n_vectors'].apply(lambda x: [item for sublist in x for item in sublist] if isinstance(x, list) else [])
            new_df['all_times_single_vector'] = new_df['all_times_n_vectors'].apply(lambda x: [item for sublist in x for item in sublist] if isinstance(x, list) else [])

            # Append to list
            results_list.append(new_df)

# Concatenate all dataframes into a single DataFrame
results = pd.concat(results_list, ignore_index=True)

# Display the results
#print(results.head(100))

In [5]:
# =====================================================================================
# PAIRED-GAP CONSOLIDATION  ->  builds two DataFrames used by every table/plot below
# -------------------------------------------------------------------------------------
# This cell computes all the paired statistics ONCE and stores them, so the table and
# plot cells that follow only *read* from these structures (no statistics are recomputed
# downstream). Run this once, after the CSV loader cell that builds `results` above.
#
# Pairing rationale: within each instance family the 20 test episodes are the SAME
# demand realisations for every policy (common random numbers), and each episode is
# compared to the PHS of that same episode. The per-episode gap
#       gap_e = (cost_pi,e - cost_phs,e) / cost_phs,e * 100        (cost = -reward)
# is therefore paired, so between-instance difficulty cancels and the mean gap has a
# far smaller standard error than its raw SD.
#
# Two DataFrames are produced:
#   results      : one row per (policy, family)  - detailed, per-family gap vector + CI
#   results_agg  : one row per policy            - pooled vectors over all 32 families,
#                  with BOTH a pooled and a family-wise mean/CI so tables & plots can
#                  choose which to display.
#
# PHS is included in both (its gap vector is 0 by construction - useful to verify).
# =====================================================================================
import numpy as np
import pandas as pd
import ast
from scipy import stats

CONF_LEVEL = 0.95   # confidence level for all CIs built here


def _as_reward_array(x):
    """AllRewards: flat list of per-episode total rewards -> 1-D float array."""
    if isinstance(x, str):
        return np.asarray(ast.literal_eval(x), dtype=float)
    return np.asarray(x, dtype=float)


def _as_time_matrix(x):
    """AllTimes: nested list (episodes x per-order decision times). Returns list of
    1-D arrays (one per episode). Uses eval with np in scope because some CSVs contain
    np.float64(...) tokens."""
    if isinstance(x, str):
        parsed = eval(x, {"np": np, "__builtins__": {}})
    else:
        parsed = x
    return [np.asarray(ep, dtype=float) for ep in parsed]


def _ci_half(vec, conf=CONF_LEVEL):
    """t-based CI half-width for the mean of `vec`."""
    v = np.asarray(vec, dtype=float)
    n = len(v)
    if n < 2:
        return np.nan
    tcrit = stats.t.ppf(0.5 + conf / 2.0, n - 1)
    return tcrit * v.std(ddof=1) / np.sqrt(n)


# ------------------------------------------------------------------ detailed: results
# `results` already exists from the loader cell (one row per policy per family, with
# an `all_rewards` column). We add the paired-gap columns to it in place.
family_cols = ['num_warehouses', 'num_customers', 'capacity_distribution']

def _row_gap_vector(r):
    fam = results[(results['num_warehouses'] == r['num_warehouses']) &
                  (results['num_customers'] == r['num_customers']) &
                  (results['capacity_distribution'] == r['capacity_distribution'])]

    #Transform rewards to costs (negative rewards) and compute the per-episode gap vector
    phs_cost = -_as_reward_array(fam[fam['policy'] == 'perfect_hindsight']['all_rewards'].iloc[0])
    cost_pi = -_as_reward_array(r['all_rewards'])
    
    assert len(cost_pi) == len(phs_cost), "episodes not aligned - cannot pair"
    assert (r['policy'] == 'perfect_hindsight') or ((cost_pi - phs_cost) >= -1e-6).all(), \
        f"PHS not per-episode optimal vs {r['policy']} - episodes misaligned"
    return (cost_pi - phs_cost) / phs_cost * 100.0

results['gap_vector'] = results.apply(_row_gap_vector, axis=1)
results['mean_gap'] = results['gap_vector'].apply(lambda g: float(np.mean(g)))
results['gap_ci_half'] = results['gap_vector'].apply(_ci_half)
results['gap_se'] = results['gap_vector'].apply(lambda g: float(np.std(g, ddof=1) / np.sqrt(len(g))))

results['mean_order_time'] = results['all_times_single_vector'].apply(
    lambda v: float(np.mean(v)) if len(v) else np.nan)


# --------------------------------------------------------- gap to exact value function
# The exact value function (EVF) policy is solved by exhaustive backward induction
# (training/train_exact_value_function.py) - its state space is exponential in total
# facility capacity, so it is only tractable for the num_customers == 50 families.
# Compute the same paired-gap statistic as above but against EVF instead of PHS, only
# for those families; all other rows get NaN. Unlike PHS, EVF is not a per-episode
# hindsight bound (it is only Bellman-optimal in expectation under a discretized
# customer-location grid), so no "cost_pi >= cost_evf" assertion applies here.
def _row_gap_vector_evf(r):
    if r['num_customers'] != 50:
        return np.nan

    fam = results[(results['num_warehouses'] == r['num_warehouses']) &
                  (results['num_customers'] == r['num_customers']) &
                  (results['capacity_distribution'] == r['capacity_distribution'])]

    evf_rows = fam[fam['policy'] == 'exact_value_function']
    if evf_rows.empty:
        return np.nan

    evf_cost = -_as_reward_array(evf_rows['all_rewards'].iloc[0])
    cost_pi = -_as_reward_array(r['all_rewards'])

    assert len(cost_pi) == len(evf_cost), "episodes not aligned - cannot pair"
    return (cost_pi - evf_cost) / evf_cost * 100.0

results['gap_vector_evf'] = results.apply(_row_gap_vector_evf, axis=1)
results['mean_gap_evf'] = results['gap_vector_evf'].apply(
    lambda g: float(np.mean(g)) if isinstance(g, np.ndarray) else np.nan)
results['gap_ci_half_evf'] = results['gap_vector_evf'].apply(
    lambda g: _ci_half(g) if isinstance(g, np.ndarray) else np.nan)
results['gap_se_evf'] = results['gap_vector_evf'].apply(
    lambda g: float(np.std(g, ddof=1) / np.sqrt(len(g))) if isinstance(g, np.ndarray) else np.nan)


# ------------------------------------------------------------- aggregated: results_agg
# One row per policy. Pool the paired gaps (20 x 32 = 640) and all per-order decision
# times (20 x 50 x 32 for that policy) across every family. Store BOTH a pooled mean/CI
# (over the flat vector) and a family-wise mean/CI (mean of the 32 per-family means).
agg_rows = []
for policy, pol_df in results.groupby('policy'):
    # pooled gap vector (concatenate the per-family 20-vectors)
    all_gaps = np.concatenate([np.asarray(g, dtype=float) for g in pol_df['gap_vector']])
    # family-wise gap means (one per family)
    fam_gap_means = np.asarray([float(np.mean(g)) for g in pol_df['gap_vector']], dtype=float)

    # pooled per-order time vector, and family-wise mean order times
    all_times = np.concatenate([np.asarray(t, dtype=float) for t in pol_df['all_times_single_vector']]) \
        if len(pol_df) else np.asarray([])
    fam_time_means = np.asarray([float(np.mean(t)) for t in pol_df['all_times_single_vector']], dtype=float)

    agg_rows.append({
        'policy': policy,
        'all_gaps': all_gaps,                      # 640-vector (20 x 32)
        'all_times': all_times,                    # pooled per-order decision times
        'fam_gap_means': fam_gap_means,            # 32-vector
        'fam_time_means': fam_time_means,          # 32-vector
        # --- pooled summaries (over the flat 640 gaps) ---
        'mean_gap_pooled': float(np.mean(all_gaps)),
        'gap_ci_half_pooled': _ci_half(all_gaps),
        # --- family-wise summaries (over the 32 family means) ---
        'mean_gap_familywise': float(np.mean(fam_gap_means)),
        'gap_ci_half_familywise': _ci_half(fam_gap_means),
        # --- runtime summaries (both flavours), in same raw units as MeanTime ---
        'mean_time_pooled': float(np.mean(all_times)) if len(all_times) else np.nan,
        'time_ci_half_pooled': _ci_half(all_times),
        'mean_time_familywise': float(np.mean(fam_time_means)) if len(fam_time_means) else np.nan,
        'time_ci_half_familywise': _ci_half(fam_time_means),
    })

results_agg = pd.DataFrame(agg_rows)

# quick sanity print (PHS gap should be ~0)
_phs = results_agg[results_agg['policy'] == 'perfect_hindsight']
print("Consolidation done.",
      f"results: {len(results)} rows (policy x family);",
      f"results_agg: {len(results_agg)} rows (per policy).")
if not _phs.empty:
    print(f"  PHS pooled mean gap (should be ~0): {_phs['mean_gap_pooled'].iloc[0]:.6f}")

Consolidation done. results: 104 rows (policy x family); results_agg: 13 rows (per policy).
  PHS pooled mean gap (should be ~0): 0.000000


In [7]:
# =====================================================================================
# TABLE(S): paired gap to PHS, per instance family + bottom Average/Min/Max rows
# -------------------------------------------------------------------------------------
# Reads ONLY from `results` (per-family gap) and `results_agg` (per-policy overall),
# both built by the consolidation cell. Emits two LaTeX tables:
#   (1) mean paired gap per family (siunitx S-columns, non-rotated), with bottom
#       Average / Min / Max rows;
#   (2) the same data, rotated/resized, with each cell also showing the 95% CI
#       half-width, plus the same bottom Average / Min / Max rows.
#
# AVG_FLAVOUR selects how the bottom "Average" row is computed:
#   'pooled'      -> mean over the flat 640 paired gaps            (results_agg.mean_gap_pooled)
#   'familywise'  -> mean of the 32 per-family mean gaps           (results_agg.mean_gap_familywise)
# The per-family cells are identical either way (they are single-family means); only the
# Average row changes. Min/Max are the smallest/largest per-family mean_gap for that
# policy across all 32 families (paired with that family's own CI half-width for the
# CI table).
# =====================================================================================

AVG_FLAVOUR = 'familywise'   # 'pooled' or 'familywise'

# column order + display labels (PHS excluded from the comparison columns)
col_policies = ['imitation_learning', 'myopic', 'genetic_programming',
                'linear_value_function_approximation', 'deep_q_networks',
                'point_estimate_lookahead', 'distributional_estimate_lookahead',
                'linear_programming_heuristic', 'linear_programming_exact',
                'parameterized_lookahead_approximation', 'proximal_policy_optimization']

_header = r"""    \multicolumn{3}{c|}{\textbf{Instance description}}
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{myo}} & \textbf{\gls{gp}} & \textbf{\gls{lvfa}} & \textbf{\gls{dqn}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{lpe}} & \textbf{\gls{pla}} & \textbf{PPO}\\
"""

_header_plain = r"""    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{myo}} & \textbf{\gls{gp}} & \textbf{\gls{lvfa}} & \textbf{\gls{dqn}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{lpe}} & \textbf{\gls{pla}} & \textbf{PPO}\\
"""


def _cell_gap(w, c, d, policy):
    row = results[(results.num_warehouses == w) & (results.num_customers == c) &
                  (results.capacity_distribution == d) & (results.policy == policy)]
    return float(row['mean_gap'].iloc[0]), float(row['gap_ci_half'].iloc[0])


def _avg_gap(policy):
    r = results_agg[results_agg.policy == policy].iloc[0]
    if AVG_FLAVOUR == 'pooled':
        return float(r['mean_gap_pooled']), float(r['gap_ci_half_pooled'])
    else:
        return float(r['mean_gap_familywise']), float(r['gap_ci_half_familywise'])


def _minmax_gap(policy, kind):
    sub = results[results.policy == policy]
    idx = sub['mean_gap'].idxmin() if kind == 'min' else sub['mean_gap'].idxmax()
    row = sub.loc[idx]
    return float(row['mean_gap']), float(row['gap_ci_half'])


def _family_rows(with_ci):
    rows = ""
    for num_customers in num_customers_options:
        rows += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
        for num_warehouses in num_warehouses_options:
            for i, cap in enumerate(capacity_distribution_options):
                if i == 0:
                    rows += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {cap.capitalize()} & "
                else:
                    rows += f"& & {cap.capitalize()} & "
                vals = {p: _cell_gap(num_warehouses, num_customers, cap, p) for p in col_policies}
                best = min(vals, key=lambda p: vals[p][0])
                cells = []
                for p in col_policies:
                    m, h = vals[p]
                    s = f"{m:.2f} ({h:.2f})" if with_ci else f"{m:.2f}"
                    cells.append(f"\\textbf{{{s}}}" if p == best else s)
                rows += " & ".join(cells) + " \\\\\n"
        if num_customers != num_customers_options[-1]:
            rows += "        \\hline\n"
    return rows


def _summary_row(label, values, with_ci):
    best = min(values, key=lambda p: values[p][0])
    cells = []
    for p in col_policies:
        m, h = values[p]
        s = f"{m:.2f} ({h:.2f})" if with_ci else f"{m:.2f}"
        cells.append(f"\\textbf{{{s}}}" if p == best else s)
    return f"\\multicolumn{{3}}{{c|}}{{{label}}} & " + " & ".join(cells) + " \\\\\n"


def _build_gap_table_plain():
    avg = {p: _avg_gap(p) for p in col_policies}
    mn = {p: _minmax_gap(p, 'min') for p in col_policies}
    mx = {p: _minmax_gap(p, 'max') for p in col_policies}
    t = r"""
\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy comparison across all families of instances: mean percentage paired gap to the \gls{phs}. For each instance family, every policy is evaluated on the same demand realisations and each run is compared to the \gls{phs} of that same instance, so the gaps are paired. Each cell reports the mean paired gap over that family's episodes. The bottom summary rows report, for each policy, the average of its per-family mean gaps (equal weight per family), and the minimum and maximum per-family mean gap. The best policy in each row is in bold.}
    \label{tab:policy_comparison_performance}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
""" + _header_plain + r"""        \toprule
""" + _family_rows(with_ci=False) \
    + "        \\midrule\n" \
        + _summary_row("Average", avg, with_ci=False) \
        + _summary_row("Min", mn, with_ci=False) \
        + _summary_row("Max", mx, with_ci=False) \
        + r"""    \bottomrule
    \end{tabular}
\end{table}
"""
    return t


def _build_gap_table_ci():
    avg = {p: _avg_gap(p) for p in col_policies}
    mn = {p: _minmax_gap(p, 'min') for p in col_policies}
    mx = {p: _minmax_gap(p, 'max') for p in col_policies}
    t = r"""
\begin{table}[!b]
    \centering
    \rotatebox{90}{%
        \begin{minipage}{0.94\textheight}
        \centering
        \caption{\scriptsize Policy comparison across all families of instances: mean percentage paired gap to the \gls{phs}, with 95\% confidence intervals. Within each family, all policies are evaluated on the same demand realiZations and each run is compared to the \gls{phs} of that same instance, so the gaps are paired. Each cell reports the mean per-instance paired gap over that family's instances; the value in parentheses is the 95\% confidence half-width. The bottom summary rows report, for each policy, the average of its per-family mean gaps (equal weight per family) with the corresponding 95\% confidence interval computed across the 32 families, together with the minimum and maximum per-family mean gap. This interval therefore reflects the variability of the mean gap across instance families rather than the precision of a single-family estimate. The minimum and maximum per-family performance are also presented. The best policy in each row is in bold.}
        \label{tab:policy_comparison_performance_ci}
        \resizebox{0.96\textheight}{!}{%
        \setlength{\tabcolsep}{8pt}
        \begin{tabular}{ccc|c|cc|cc|cc|cccc}
    \toprule
""" + _header + r"""        \toprule
""" + _family_rows(with_ci=True) \
        + "        \\midrule\n" \
        + _summary_row("\\textbf{Average}", avg, with_ci=True) \
        + _summary_row("\\textbf{Min}", mn, with_ci=True) \
        + _summary_row("\\textbf{Max}", mx, with_ci=True) \
        + r"""        \bottomrule

        \end{tabular}%
        }
        \end{minipage}%
    }
\end{table}
"""
    return t


print("% ===== MAIN paired-gap table (means only, Average/Min/Max) =====")
print(_build_gap_table_plain())
print("\n% ===== paired-gap table WITH 95% CI (Average/Min/Max) =====")
print(_build_gap_table_ci())

% ===== MAIN paired-gap table (means only, Average/Min/Max) =====

\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy comparison across all families of instances: mean percentage paired gap to the \gls{phs}. For each instance family, every policy is evaluated on the same demand realisations and each run is compared to the \gls{phs} of that same instance, so the gaps are paired. Each cell reports the mean paired gap over that family's episodes. The bottom summary rows report, for each policy, the average of its per-family mean gaps (equal weight per family), and the minimum and maximum per-family mean gap. The best policy in each row is in bold.}
    \label{tab:policy_comparison_performance}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf

In [4]:
# =====================================================================================
# TABLE: paired gap to EVF (exact value function), 50-customer families only
# -------------------------------------------------------------------------------------
# Same construction as the PHS gap table above, but benchmarked against the exact value
# function (EVF) policy instead of PHS, using the `mean_gap_evf` / `gap_ci_half_evf`
# columns added in the consolidation cell. EVF is only tractable for num_customers ==
# 50 (exhaustive backward induction over remaining capacity), so this table covers just
# those 8 families (4 warehouse counts x 2 capacity distributions) instead of all 32.
# EVF itself is excluded from the comparison columns (its gap to itself is 0 by
# construction), the same treatment PHS gets in the table above. Reuses col_policies,
# _header/_header_plain and _summary_row from the cell above.
# =====================================================================================

EVF_NUM_CUSTOMERS = 50
_results_evf = results[(results['num_customers'] == EVF_NUM_CUSTOMERS) &
                        (results['mean_gap_evf'].notna())]


def _cell_gap_evf(w, d, policy):
    row = _results_evf[(_results_evf.num_warehouses == w) &
                        (_results_evf.capacity_distribution == d) &
                        (_results_evf.policy == policy)]
    return float(row['mean_gap_evf'].iloc[0]), float(row['gap_ci_half_evf'].iloc[0])


def _avg_gap_evf(policy):
    sub = _results_evf[_results_evf.policy == policy]
    return float(sub['mean_gap_evf'].mean()), _ci_half(sub['mean_gap_evf'])


def _minmax_gap_evf(policy, kind):
    sub = _results_evf[_results_evf.policy == policy]
    idx = sub['mean_gap_evf'].idxmin() if kind == 'min' else sub['mean_gap_evf'].idxmax()
    row = sub.loc[idx]
    return float(row['mean_gap_evf']), float(row['gap_ci_half_evf'])


def _family_rows_evf(with_ci):
    rows = f"        \\multirow{{8}}{{*}}{{{EVF_NUM_CUSTOMERS}}} "
    for num_warehouses in num_warehouses_options:
        for i, cap in enumerate(capacity_distribution_options):
            if i == 0:
                rows += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {cap.capitalize()} & "
            else:
                rows += f"& & {cap.capitalize()} & "
            vals = {p: _cell_gap_evf(num_warehouses, cap, p) for p in col_policies}
            best = min(vals, key=lambda p: vals[p][0])
            cells = []
            for p in col_policies:
                m, h = vals[p]
                s = f"{m:.2f} ({h:.2f})" if with_ci else f"{m:.2f}"
                cells.append(f"\\textbf{{{s}}}" if p == best else s)
            rows += " & ".join(cells) + " \\\\\n"
    return rows


def _build_gap_table_evf_plain():
    avg = {p: _avg_gap_evf(p) for p in col_policies}
    mn = {p: _minmax_gap_evf(p, 'min') for p in col_policies}
    mx = {p: _minmax_gap_evf(p, 'max') for p in col_policies}
    t = r"""
\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy comparison for the 50-customer instance families: mean percentage paired gap to the optimal policy, solved by exhaustive backward induction. For each instance family, every policy is evaluated on the same demand realizations and each run is compared to the optimal policy for that same instance, so the gaps are paired. Each cell reports the mean paired gap over that family's episodes. The bottom summary rows report, for each policy, the average of its per-family mean gaps (equal weight per family), and the minimum and maximum per-family mean gap. The best policy in each row is in bold.}
    \label{tab:policy_comparison_performance_evf}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
""" + _header_plain + r"""        \toprule
""" + _family_rows_evf(with_ci=False) \
    + "        \\midrule\n" \
        + _summary_row("Average", avg, with_ci=False) \
        + _summary_row("Min", mn, with_ci=False) \
        + _summary_row("Max", mx, with_ci=False) \
        + r"""    \bottomrule
    \end{tabular}
\end{table}
"""
    return t


def _build_gap_table_evf_ci():
    avg = {p: _avg_gap_evf(p) for p in col_policies}
    mn = {p: _minmax_gap_evf(p, 'min') for p in col_policies}
    mx = {p: _minmax_gap_evf(p, 'max') for p in col_policies}
    t = r"""
\begin{table}[!b]
    \centering
    \rotatebox{90}{%
        \begin{minipage}{0.94\textheight}
        \centering
        \caption{\scriptsize Policy comparison for the 50-customer instance families: mean percentage paired gap to the optimal policy, with 95\% confidence intervals. Within each family, all policies are evaluated on the same demand realizations and each run is compared to the optimal policy for that same instance, so the gaps are paired. Each cell reports the mean per-instance paired gap over that family's instances; the value in parentheses is the 95\% confidence half-width. The bottom summary rows report, for each policy, the average of its per-family mean gaps (equal weight per family) with the corresponding 95\% confidence interval computed across the 8 families, together with the minimum and maximum per-family mean gap. The best policy in each row is in bold.}
        \label{tab:policy_comparison_performance_evf_ci}
        \resizebox{0.96\textheight}{!}{%
        \setlength{\tabcolsep}{8pt}
        \begin{tabular}{ccc|c|cc|cc|cc|cccc}
    \toprule
""" + _header + r"""        \toprule
""" + _family_rows_evf(with_ci=True) \
        + "        \\midrule\n" \
        + _summary_row("\\textbf{Average}", avg, with_ci=True) \
        + _summary_row("\\textbf{Min}", mn, with_ci=True) \
        + _summary_row("\\textbf{Max}", mx, with_ci=True) \
        + r"""        \bottomrule

        \end{tabular}%
        }
        \end{minipage}%
    }
\end{table}
"""
    return t


print("% ===== EVF paired-gap table (50-customer families only, means only, Average/Min/Max) =====")
print(_build_gap_table_evf_plain())
print("\n% ===== EVF paired-gap table (50-customer families only) WITH 95% CI =====")
print(_build_gap_table_evf_ci())


% ===== EVF paired-gap table (50-customer families only, means only, Average/Min/Max) =====

\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy comparison for the 50-customer instance families: mean percentage paired gap to the optimal policy, solved by exhaustive backward induction. For each instance family, every policy is evaluated on the same demand realizations and each run is compared to the optimal policy for that same instance, so the gaps are paired. Each cell reports the mean paired gap over that family's episodes. The bottom summary rows report, for each policy, the average of its per-family mean gaps (equal weight per family), and the minimum and maximum per-family mean gap. The best policy in each row is in bold.}
    \label{tab:policy_comparison_performance_evf}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2

In [ ]:
# =====================================================================================
# TABLE: all-vs-all paired comparison matrix, combined (mean gap diff + stacked p-value)
# -------------------------------------------------------------------------------------
# Self-contained: reads only `results` (per-family gap_vector, from the consolidation
# cell) and recomputes everything it needs. For each ordered pair (row A, col B) we
# take, per family, the mean paired gap difference d_fam = mean_e(gap_A,e - gap_B,e)
# (one scalar per family), then average and t-test those 32 family-level differences
# (two-stage: every family has equal weight, n = 32, avoiding pseudo-replication from
# treating all 640 episodes as independent).
#
# Each cell stacks the mean gap difference (with significance stars, top) over its
# paired t-test p-value (bottom, in parentheses), via \shortstack, so a single table
# replaces the separate mean-diff and p-value matrices.
#   negative => row A has a smaller gap => A is BETTER than B.
# PHS is excluded (it is the reference, not a competitor).
# =====================================================================================
import numpy as np
from scipy import stats

matrix_policies = ['imitation_learning', 'myopic', 'genetic_programming',
                   'linear_value_function_approximation', 'deep_q_networks',
                   'point_estimate_lookahead', 'distributional_estimate_lookahead',
                   'linear_programming_heuristic', 'linear_programming_exact',
                   'parameterized_lookahead_approximation', 'proximal_policy_optimization']
matrix_labels = {'myopic': 'MYO', 'imitation_learning': 'IL', 'genetic_programming': 'GP',
                 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL',
                 'linear_value_function_approximation': 'LVFA', 'deep_q_networks': 'DQN',
                 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE',
                 'parameterized_lookahead_approximation': 'PLA', 'proximal_policy_optimization': 'PPO'}
labels = [matrix_labels[p] for p in matrix_policies]

fam_keys = list(results.groupby(['num_warehouses', 'num_customers', 'capacity_distribution']).groups.keys())


def _gap_vec(policy, key):
    w, c, d = key
    row = results[(results.num_warehouses == w) & (results.num_customers == c) &
                  (results.capacity_distribution == d) & (results.policy == policy)]
    return np.asarray(row['gap_vector'].iloc[0], dtype=float)


def _family_gap_diff_vector(a, b):
    return np.array([(_gap_vec(a, k) - _gap_vec(b, k)).mean() for k in fam_keys])


def _stars(p):
    if np.isnan(p):
        return ""
    if p < 0.01:
        return "^{**}"
    if p < 0.05:
        return "^{*}"
    return ""


n_pol = len(matrix_policies)
mean_diff = np.full((n_pol, n_pol), np.nan)
p_ttest = np.full((n_pol, n_pol), np.nan)

for ia, a in enumerate(matrix_policies):
    for ib, b in enumerate(matrix_policies):
        if ia == ib:
            continue
        dv = _family_gap_diff_vector(a, b)
        mean_diff[ia, ib] = dv.mean()
        _, p_ttest[ia, ib] = stats.ttest_1samp(dv, 0.0)

col_fmt = "l|" + "r" * n_pol

mc = r"""
\begin{table}[h!]
    \centering
    \scriptsize
    \renewcommand{\arraystretch}{1.35}
    \setlength{\tabcolsep}{4pt}
    \caption{Pairwise paired comparison of policies' performance across the 32 families of instances. Each cell stacks the mean, over families, of the paired gap-to-\gls{phs} difference (row policy $-$ column policy), in percentage points, with its paired $t$-test $p$-value below in parentheses; negative values mean the row policy has a smaller gap to the \gls{phs}, and therefore better performance. The \gls{myo} column (and row) gives every policy's difference in gap relative to the myopic baseline. Significance stars: $^{*}\,p<0.05$, $^{**}\,p<0.01$.}
    \begin{tabular}{""" + col_fmt + r"""}
    \toprule
    \textbf{Row $-$ Col} & """ + " & ".join(f"\\textbf{{{x}}}" for x in labels) + r""" \\
    \midrule
"""
for ia in range(n_pol):
    cells = [f"\\textbf{{{labels[ia]}}}"]
    for ib in range(n_pol):
        if ia == ib:
            cells.append("")
        else:
            top = f"${mean_diff[ia, ib]:.2f}{_stars(p_ttest[ia, ib])}$"
            bot = f"({p_ttest[ia, ib]:.3f})"
            cells.append(f"\\shortstack{{{top}\\\\{bot}}}")
    mc += "    " + " & ".join(cells) + r" \\" + "\n"
mc += r"""    \bottomrule
    \end{tabular}
\end{table}
"""
print("% ===== pairwise paired matrix, combined (mean gap diff over p-value) =====")
print(mc)


% ===== pairwise paired matrix, combined (mean gap diff over p-value) =====

\begin{table}[h!]
    \centering
    \scriptsize
    \renewcommand{\arraystretch}{1.35}
    \setlength{\tabcolsep}{4pt}
    \caption{Pairwise paired comparison of policies' performance across the 32 families of instances. Each cell stacks the mean, over families, of the paired gap-to-\gls{phs} difference (row policy $-$ column policy), in percentage points, with its paired $t$-test $p$-value below in parentheses; negative values mean the row policy has a smaller gap to the \gls{phs}, and therefore better performance. The \gls{myo} column (and row) gives every policy's difference in gap relative to the myopic baseline. Significance stars: $^{*}\,p<0.05$, $^{**}\,p<0.01$.}
    \begin{tabular}{l|rrrrrrrrrrr}
    \toprule
    \textbf{Row $-$ Col} & \textbf{IL} & \textbf{MYO} & \textbf{GP} & \textbf{LVFA} & \textbf{DQN} & \textbf{PEL} & \textbf{DEL} & \textbf{LPH} & \textbf{LPE} & \textbf{PLA} & \textbf{PPO} \\
   

In [ ]:
# =====================================================================================
# TABLE: all-vs-all paired comparison matrix, combined (order runtime, in microseconds)
# -------------------------------------------------------------------------------------
# Same construction as the gap-comparison matrix above, but on per-order decision time
# instead of paired gap. Self-contained: reads only `results` (per-family
# all_times_single_vector, from the consolidation cell - one flat vector of per-order
# decision times per (policy, family), in nanoseconds, ordered episode-major then
# order-within-episode; policies share the same 20 CRN episodes and per-episode order
# arrival sequence within a family, so elementwise pairing across policies is valid).
#
# For each ordered pair (row A, col B) we take, per family, the mean paired runtime
# difference d_fam = mean_o(time_A,o - time_B,o) in microseconds (one scalar per
# family), then average and t-test those 32 family-level differences (two-stage,
# n = 32, same rationale as the gap matrix).
#
# Each cell stacks the mean runtime difference (with significance stars, top) over its
# paired t-test p-value (bottom, in parentheses).
#   negative => row A is FASTER (smaller per-order decision time) than B.
# PHS is excluded (it has no real "decision" - the action is read off the known future).
# =====================================================================================
import numpy as np
from scipy import stats

matrix_policies = ['imitation_learning', 'myopic', 'genetic_programming',
                   'linear_value_function_approximation', 'deep_q_networks',
                   'point_estimate_lookahead', 'distributional_estimate_lookahead',
                   'linear_programming_heuristic', 'linear_programming_exact',
                   'parameterized_lookahead_approximation', 'proximal_policy_optimization']
matrix_labels = {'myopic': 'MYO', 'imitation_learning': 'IL', 'genetic_programming': 'GP',
                 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL',
                 'linear_value_function_approximation': 'LVFA', 'deep_q_networks': 'DQN',
                 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE',
                 'parameterized_lookahead_approximation': 'PLA', 'proximal_policy_optimization': 'PPO'}
labels = [matrix_labels[p] for p in matrix_policies]

fam_keys = list(results.groupby(['num_warehouses', 'num_customers', 'capacity_distribution']).groups.keys())

NS_TO_US = 1.0 / 1_000.0


def _time_vec(policy, key):
    w, c, d = key
    row = results[(results.num_warehouses == w) & (results.num_customers == c) &
                  (results.capacity_distribution == d) & (results.policy == policy)]
    return np.asarray(row['all_times_single_vector'].iloc[0], dtype=float) * NS_TO_US


def _family_time_diff_vector(a, b):
    return np.array([(_time_vec(a, k) - _time_vec(b, k)).mean() for k in fam_keys])


def _stars(p):
    if np.isnan(p):
        return ""
    if p < 0.01:
        return "^{**}"
    if p < 0.05:
        return "^{*}"
    return ""


n_pol = len(matrix_policies)
mean_time_diff = np.full((n_pol, n_pol), np.nan)
p_ttest_time = np.full((n_pol, n_pol), np.nan)

for ia, a in enumerate(matrix_policies):
    for ib, b in enumerate(matrix_policies):
        if ia == ib:
            continue
        dv = _family_time_diff_vector(a, b)
        mean_time_diff[ia, ib] = dv.mean()
        _, p_ttest_time[ia, ib] = stats.ttest_1samp(dv, 0.0)

col_fmt = "l|" + "r" * n_pol

mt = r"""
\begin{table}[h!]
    \centering
    \tiny
    \renewcommand{\arraystretch}{1.35}
    \setlength{\tabcolsep}{1pt}
    \caption{Pairwise paired comparison of policies' online runtime across the 32 families of instances. Each cell stacks the mean, over families, of the per-order paired decision-time difference (row policy $-$ column policy), in microseconds, with its paired $t$-test $p$-value below in parentheses; negative values mean the row policy is faster than the column policy. The \gls{myo} column (and row) gives every policy's runtime relative to the myopic baseline. Significance stars: $^{*}\,p<0.05$, $^{**}\,p<0.01$.}
    \label{tab:pairwise_paired_matrix_time_combined}
    \begin{tabular}{""" + col_fmt + r"""}
    \toprule
    \textbf{Row $-$ Col} & """ + " & ".join(f"\\textbf{{{x}}}" for x in labels) + r""" \\
    \midrule
"""
for ia in range(n_pol):
    cells = [f"\\textbf{{{labels[ia]}}}"]
    for ib in range(n_pol):
        if ia == ib:
            cells.append("")
        else:
            top = f"${mean_time_diff[ia, ib]:.2f}{_stars(p_ttest_time[ia, ib])}$"
            bot = f"({p_ttest_time[ia, ib]:.3f})"
            cells.append(f"\\shortstack{{{top}\\\\{bot}}}")
    mt += "    " + " & ".join(cells) + r" \\" + "\n"
mt += r"""    \bottomrule
    \end{tabular}
\end{table}
"""
print("% ===== pairwise paired matrix, combined (order runtime diff over p-value, microseconds) =====")
print(mt)


% ===== pairwise paired matrix, combined (order runtime diff over p-value, microseconds) =====

\begin{table}[h!]
    \centering
    \tiny
    \renewcommand{\arraystretch}{1.35}
    \setlength{\tabcolsep}{1pt}
    \caption{Pairwise paired comparison of policies' online runtime across the 32 families of instances. Each cell stacks the mean, over families, of the per-order paired decision-time difference (row policy $-$ column policy), in microseconds, with its paired $t$-test $p$-value below in parentheses; negative values mean the row policy is faster than the column policy. The \gls{myo} column (and row) gives every policy's runtime relative to the myopic baseline. Significance stars: $^{*}\,p<0.05$, $^{**}\,p<0.01$.}
    \label{tab:pairwise_paired_matrix_time_combined}
    \begin{tabular}{l|rrrrrrrrrrr}
    \toprule
    \textbf{Row $-$ Col} & \textbf{IL} & \textbf{MYO} & \textbf{GP} & \textbf{LVFA} & \textbf{DQN} & \textbf{PEL} & \textbf{DEL} & \textbf{LPH} & \textbf{LPE} & \textbf{P

In [ ]:

policies = ['imitation_learning', 'myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'point_estimate_lookahead', 'distributional_estimate_lookahead', 'linear_programming_heuristic','linear_programming_exact','parameterized_lookahead_approximation', 'proximal_policy_optimization']
policies_dict = {'myopic': 'MYO', 'imitation_learning': 'IL', 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL', 'parameterized_lookahead_approximation': 'CFA-DLA', 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE', 'proximal_policy_optimization': 'PPO'}


latex_table = r"""
\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy comparison across all families of instances: mean online runtime per order, in milliseconds. Each cell reports the mean runtime over that family's episodes. The bottom summary rows report, for each policy, the average of its per-family mean runtimes (equal weight per family), and the minimum and maximum per-family mean runtime. Columns marked with * correspond to very low runtime values and are multiplied by $10^3$; the values in these columns can therefore be read directly as microseconds. The fastest policy in each row is in bold.}
    \label{tab:policy_comparison_time}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}\textsuperscript{*}} & \textbf{\gls{myo}\textsuperscript{*}} & \textbf{\gls{gp}\textsuperscript{*}} & \textbf{\gls{lvfa}\textsuperscript{*}} & \textbf{\gls{dqn}\textsuperscript{*}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{lpe}} & \textbf{\gls{pla}} & \textbf{\gls{ppo}\textsuperscript{*}}\\
        \toprule
"""

# Loop through the customer and warehouse options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "
            
            # Filter the DataFrame for the current configuration
            df_filtered = results[
                (results["num_customers"] == num_customers) & 
                (results["num_warehouses"] == num_warehouses) & 
                (results["capacity_distribution"] == capacity_distribution)
            ]
            
            # Extract the percentage above perfect hindsight for each policy
            times = []
            short_times = []
            for policy in policies:
                time = df_filtered.loc[df_filtered['policy'] == policy, 'mean_time'].values[0]
                time = time / 1_000
                if policy not in ['myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']:
                    time = time / 1_000  # Convert to milliseconds
                else:
                    short_times.append(time)
                times.append(time)

            # Find the min time in times, knowing that short_times contains the relevant ones
            min_time = min(short_times)

            # Format the percentages, highlighting the minimum in bold
            formatted_times = [
                f"\\textbf{{{time:.2f}}}" if time == min_time else f"{time:.2f}"
                for time in times
            ]
            
            latex_table += " & ".join(formatted_times) + r" \\" + "\n"
        #latex_table += f"        \\cline{{2-{len(policies)+3}}}\n"
    latex_table += "        \\bottomrule\n"

# Add average, min, and max rows
all_min_time = []
all_mean_time = []
all_max_time = []   
for policy in policies:
    all_min_time.append(results.loc[results['policy'] == policy, 'mean_time'].min())
    all_mean_time.append(results.loc[results['policy'] == policy, 'mean_time'].mean())
    all_max_time.append(results.loc[results['policy'] == policy, 'mean_time'].max())

short_time_policies = ['myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']

def convert_time(value, policy):
    value = value / 1_000
    if policy not in short_time_policies:
        value = value / 1_000  # Convert to milliseconds
    return value

converted_mean_time = [convert_time(v, p) for v, p in zip(all_mean_time, policies)]
converted_min_time = [convert_time(v, p) for v, p in zip(all_min_time, policies)]
converted_max_time = [convert_time(v, p) for v, p in zip(all_max_time, policies)]

best_mean_short = min(v for v, p in zip(converted_mean_time, policies) if p in short_time_policies)
best_mean_long = min(v for v, p in zip(converted_mean_time, policies) if p not in short_time_policies)
best_min_short = min(v for v, p in zip(converted_min_time, policies) if p in short_time_policies)
best_min_long = min(v for v, p in zip(converted_min_time, policies) if p not in short_time_policies)
best_max_short = min(v for v, p in zip(converted_max_time, policies) if p in short_time_policies)
best_max_long = min(v for v, p in zip(converted_max_time, policies) if p not in short_time_policies)

latex_table += "\multicolumn{3}{c|}{Average} \n"
for policy, mean_value in zip(policies, converted_mean_time):
    latex_table += f"& \\textbf{{{mean_value:.2f}}}" if mean_value == best_mean_short else f"& {mean_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Min} \n"
for policy, min_value in zip(policies, converted_min_time):
    latex_table += f"& \\textbf{{{min_value:.2f}}}" if min_value == best_min_short else f"& {min_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Max} \n"
for policy, max_value in zip(policies, converted_max_time):
    latex_table += f"& \\textbf{{{max_value:.2f}}}" if max_value == best_max_short else f"& {max_value:.2f} "
latex_table += r"\\ \bottomrule"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)


\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy comparison across all families of instances: mean online runtime per order, in milliseconds. Each cell reports the mean runtime over that family's episodes. The bottom summary rows report, for each policy, the average of its per-family mean runtimes (equal weight per family), and the minimum and maximum per-family mean runtime. Columns marked with * correspond to very low runtime values and are multiplied by $10^3$; the values in these columns can therefore be read directly as microseconds. The fastest policy in each row is in bold.}
    \label{tab:policy_comparison_time}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above

<>:89: SyntaxWarning: invalid escape sequence '\m'
<>:93: SyntaxWarning: invalid escape sequence '\m'
<>:97: SyntaxWarning: invalid escape sequence '\m'
<>:89: SyntaxWarning: invalid escape sequence '\m'
<>:93: SyntaxWarning: invalid escape sequence '\m'
<>:97: SyntaxWarning: invalid escape sequence '\m'
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_48138/3997262794.py:89: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Average} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_48138/3997262794.py:93: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Min} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_48138/3997262794.py:97: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Max} \n"


In [ ]:


policies = ['imitation_learning', 'myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'point_estimate_lookahead', 'distributional_estimate_lookahead', 'linear_programming_heuristic','linear_programming_exact','parameterized_lookahead_approximation', 'proximal_policy_optimization']
policies_dict = {'myopic': 'MYO', 'imitation_learning': 'IL', 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL', 'parameterized_lookahead_approximation': 'CFA-DLA', 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE', 'proximal_policy_optimization': 'PPO'}


latex_table = r"""
\begin{table}[!b]
    \centering
    \rotatebox{90}{%
        \begin{minipage}{0.93\textheight}
        \centering
        \caption{\scriptsize Policy comparison across all families of instances: mean online runtime per order, in milliseconds, with standard deviation. Each cell reports the mean runtime over that family's episodes, with the standard deviation in parentheses. Columns marked with * correspond to very low runtime values and are multiplied by $10^3$; the values in these columns can therefore be read directly as microseconds. The fastest policy in each row is in bold.}
        \label{tab:policy_comparison_time_std}
        \resizebox{0.96\textheight}{!}{%
        %\small
        \setlength{\tabcolsep}{8pt}
        \begin{tabular}{ccc|c|cc|cc|cc|cccc}
        \toprule
        \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}\textsuperscript{*}} & \textbf{\gls{myo}\textsuperscript{*}} & \textbf{\gls{gp}\textsuperscript{*}} & \textbf{\gls{lvfa}\textsuperscript{*}} & \textbf{\gls{dqn}\textsuperscript{*}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{lpe}} & \textbf{\gls{pla}} & \textbf{\gls{ppo}\textsuperscript{*}}\\
        \toprule
"""

# Loop through the customer and warehouse options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "
            
            # Filter the DataFrame for the current configuration
            df_filtered = results[
                (results["num_customers"] == num_customers) & 
                (results["num_warehouses"] == num_warehouses) & 
                (results["capacity_distribution"] == capacity_distribution)
            ]
            
            # Extract the mean (and std) runtime for each policy
            times = []
            stds = []
            short_times = []
            for policy in policies:
                time = df_filtered.loc[df_filtered['policy'] == policy, 'mean_time'].values[0]
                std = df_filtered.loc[df_filtered['policy'] == policy, 'std_time'].values[0]
                if policy in ['myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']:
                    time = time / 1_000  # Convert to microseconds
                    std = std / 1_000
                    short_times.append(time)
                else:
                    time = time / 1_000_000  # Convert to milliseconds
                    std = std / 1_000_000

                times.append(time)
                stds.append(std)

            # Find the min time in times, knowing that short_times contains the relevant ones
            min_time = min(short_times)

            # Format the times, highlighting the minimum in bold; std shown in parentheses after the mean
            formatted_times = [
                f"\\textbf{{{time:.2f}}} ({std:.2f})" if time == min_time else f"{time:.2f} ({std:.2f})"
                for time, std in zip(times, stds)
            ]
            
            latex_table += " & ".join(formatted_times) + r" \\" + "\n"
        #latex_table += f"        \\cline{{2-{len(policies)+3}}}\n"
    latex_table += "        \\bottomrule\n"

# Add average, min, and max rows (with standard deviation alongside each)
all_min_time = []
all_mean_time = []
all_max_time = []
all_avg_std_time = []  # between-family std of the per-family means (paired with Average)
all_min_std_time = []  # within-family std of the family attaining the min mean (paired with Min)
all_max_std_time = []  # within-family std of the family attaining the max mean (paired with Max)
for policy in policies:
    policy_rows = results.loc[results['policy'] == policy]
    all_min_time.append(policy_rows['mean_time'].min())
    all_mean_time.append(policy_rows['mean_time'].mean())
    all_max_time.append(policy_rows['mean_time'].max())
    all_avg_std_time.append(policy_rows['mean_time'].std())
    all_min_std_time.append(policy_rows.loc[policy_rows['mean_time'].idxmin(), 'std_time'])
    all_max_std_time.append(policy_rows.loc[policy_rows['mean_time'].idxmax(), 'std_time'])

short_time_policies = ['myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']

def convert_time(value, policy):
    if policy in short_time_policies:
        return value / 1_000  # Convert to microseconds
    return value / 1_000_000  # Convert to milliseconds

'''
converted_mean_time = [convert_time(v, p) for v, p in zip(all_mean_time, policies)]
converted_min_time = [convert_time(v, p) for v, p in zip(all_min_time, policies)]
converted_max_time = [convert_time(v, p) for v, p in zip(all_max_time, policies)]
converted_avg_std_time = [convert_time(v, p) for v, p in zip(all_avg_std_time, policies)]
converted_min_std_time = [convert_time(v, p) for v, p in zip(all_min_std_time, policies)]
converted_max_std_time = [convert_time(v, p) for v, p in zip(all_max_std_time, policies)]

best_mean_short = min(v for v, p in zip(converted_mean_time, policies) if p in short_time_policies)
best_mean_long = min(v for v, p in zip(converted_mean_time, policies) if p not in short_time_policies)
best_min_short = min(v for v, p in zip(converted_min_time, policies) if p in short_time_policies)
best_min_long = min(v for v, p in zip(converted_min_time, policies) if p not in short_time_policies)
best_max_short = min(v for v, p in zip(converted_max_time, policies) if p in short_time_policies)
best_max_long = min(v for v, p in zip(converted_max_time, policies) if p not in short_time_policies)

latex_table += "\multicolumn{3}{c|}{Average} \n"
for policy, mean_value, std_value in zip(policies, converted_mean_time, converted_avg_std_time):
    latex_table += f"& \\textbf{{{mean_value:.2f}}} ({std_value:.2f})" if mean_value == best_mean_short else f"& {mean_value:.2f} ({std_value:.2f}) "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Min} \n"
for policy, min_value, std_value in zip(policies, converted_min_time, converted_min_std_time):
    latex_table += f"& \\textbf{{{min_value:.2f}}} ({std_value:.2f})" if min_value == best_min_short else f"& {min_value:.2f} ({std_value:.2f}) "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Max} \n"
for policy, max_value, std_value in zip(policies, converted_max_time, converted_max_std_time):
    latex_table += f"& \\textbf{{{max_value:.2f}}} ({std_value:.2f})" if max_value == best_max_short else f"& {max_value:.2f} ({std_value:.2f}) "
'''

latex_table += r"\\ \bottomrule"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
            }%
        \end{minipage}
        }
    \end{table}
"""

# Print the LaTeX table string
print(latex_table)



\begin{table}[!b]
    \centering
    \rotatebox{90}{%
        \begin{minipage}{0.93\textheight}
        \centering
        \caption{\scriptsize Policy comparison across all families of instances: mean online runtime per order, in milliseconds, with standard deviation. Each cell reports the mean runtime over that family's episodes, with the standard deviation in parentheses. The bottom summary rows report, for each policy, the average of its per-family mean runtimes (equal weight per family) with the standard deviation of those per-family means in parentheses, and the minimum and maximum per-family mean runtime, each with the standard deviation of that family's episodes in parentheses. Columns marked with * correspond to very low runtime values and are multiplied by $10^3$; the values in these columns can therefore be read directly as microseconds. The fastest policy in each row is in bold.}
        \label{tab:policy_comparison_time_std}
        \resizebox{0.96\textheight}{!}{%
       

<>:109: SyntaxWarning: invalid escape sequence '\m'
<>:113: SyntaxWarning: invalid escape sequence '\m'
<>:117: SyntaxWarning: invalid escape sequence '\m'
<>:109: SyntaxWarning: invalid escape sequence '\m'
<>:113: SyntaxWarning: invalid escape sequence '\m'
<>:117: SyntaxWarning: invalid escape sequence '\m'
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_48138/177685864.py:109: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Average} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_48138/177685864.py:113: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Min} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_48138/177685864.py:117: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Max} \n"


In [ ]:
import pandas as pd

# Read the theta.csv file
theta_df = pd.read_csv('training/linear_value_function_approximation_training/theta.csv')

# Define the LaTeX table structure
latex_table = r"""
\begin{table}[H]
    \centering
    \scriptsize
    \caption{\gls{lvfa} final weights ($\theta$) across all families of instances.}
    \label{tab:lvfa_theta_all}
    \begin{tabular}{ccc|S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta_A$} & \textbf{$\theta_B$} & \textbf{$\theta_C$} & \textbf{$\theta_D$} & \textbf{$\theta_E$} \\
        \midrule
"""

# Loop through the customer, warehouse, and capacity options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Filter the DataFrame for the current configuration
            df_filtered = theta_df[
                (theta_df["num_customers"] == num_customers) &
                (theta_df["num_warehouses"] == num_warehouses) &
                (theta_df["capacity_distribution"] == capacity_distribution)
            ]

            # Extract the theta values - one coefficient per facility, so pad with
            # blanks up to the 5-column max (num_warehouses_options tops out at 5)
            MAX_THETA_COLS = 5
            if not df_filtered.empty:
                theta = df_filtered.iloc[0]['theta']
                theta_values = [f"{round(float(t), 2)}" for t in theta.split(',')]
            else:
                theta_values = []
            theta_cells = theta_values + [""] * (MAX_THETA_COLS - len(theta_values))
            latex_table += " & ".join(theta_cells) + " \\\\\n"
    latex_table += "        \\hline\n"
latex_table += "        \\bottomrule\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)



\begin{table}[H]
    \centering
    \scriptsize
    \caption{\gls{lvfa} final weights ($\theta$) across all families of instances.}
    \label{tab:lvfa_theta_all}
    \begin{tabular}{ccc|S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta_A$} & \textbf{$\theta_B$} & \textbf{$\theta_C$} & \textbf{$\theta_D$} & \textbf{$\theta_E$} \\
        \midrule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & -36.11 & -36.05 &  &  &  \\
& & Uneven & -50.52 & -8.32 &  &  &  \\
& \multirow{2}{*}{3} & Uniform & -38.87 & -14.32 & -35.4 &  &  \\
& & Uneven & -54.42 & -6.62 & -9.02 &  &  \\
& \multirow{2}{*}{4} & Uniform & -28.38 & -28.01 & -22.68 & -22.85 &  \\
& & Uneven & -45.48 & -37.07 & -2.36 & 22.25 &  \\
& \multirow{2}{*}{5} & Uniform & -22.23 & -22.01 & -22.15 & -22.22 & -32.9 \\
& & Uneven & -43.9 & -30.52 & -12.06 & 1.83 & -

In [ ]:
# PLA (parameterized lookahead approximation) sensitivity to theta, as a table instead
# of the plot above: per instance, the mean reward at each tested theta plus the
# selected (best) theta. Best = highest mean_reward (reward is negative distance, so
# less negative is better - see Eq. 9/23 as in the plot cell above).
import pandas as pd

param_options = [1.0, 1.02, 1.04, 1.06, 1.08, 1.10]

pla_df = pd.read_csv('training/parameterized_lookahead_approximation_training/parameterized_lookahead_approximation_all_params.csv')

latex_table = r"""
\begin{table}[H]
    \scriptsize
    \centering
    \caption{\gls{pla} mean reward per tested $\theta$ and selected $\theta$ across all families of instances.}
    \label{tab:pla_theta_sensitivity}
    \begin{tabular}{ccc|S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]|S[table-format=1.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} & \multicolumn{6}{c|}{\textbf{Mean reward per tested $\theta$}} & \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta=1.00$} & \textbf{$\theta=1.02$} & \textbf{$\theta=1.04$} & \textbf{$\theta=1.06$} & \textbf{$\theta=1.08$} & \textbf{$\theta=1.10$} & \textbf{Selected $\theta$} \\
        \toprule
"""

# Loop through the customer, warehouse, and capacity options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Filter the DataFrame for the current configuration
            df_filtered = pla_df[
                (pla_df["num_customers"] == num_customers) &
                (pla_df["num_warehouses"] == num_warehouses) &
                (pla_df["capacity_distribution"] == capacity_distribution)
            ]

            # Extract the mean reward for each tested theta
            rewards = []
            for param in param_options:
                reward = df_filtered.loc[df_filtered['param'] == param, 'mean_reward'].values[0]
                rewards.append(reward)

            # Best theta = highest mean reward (least negative)
            best_reward = max(rewards)
            best_param = param_options[rewards.index(best_reward)]

            formatted_rewards = [
                f"\\textbf{{-{reward:.2f}}}" if reward == best_reward else f"{reward:.2f}"
                for reward in rewards
            ]

            latex_table += " & ".join(formatted_rewards) + f" & {best_param:.2f} " + r" \\" + "\n"
    latex_table += "        \\hline\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)



\begin{table}[H]
    \scriptsize
    \centering
    \caption{\gls{pla} mean reward per tested $\theta$ and selected $\theta$ across all families of instances.}
    \label{tab:pla_theta_sensitivity}
    \begin{tabular}{ccc|S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]|S[table-format=1.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} & \multicolumn{6}{c|}{\textbf{Mean reward per tested $\theta$}} & \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta=1.00$} & \textbf{$\theta=1.02$} & \textbf{$\theta=1.04$} & \textbf{$\theta=1.06$} & \textbf{$\theta=1.08$} & \textbf{$\theta=1.10$} & \textbf{Selected $\theta$} \\
        \toprule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & -3306.03 & -3295.14 & -3295.98 & -3289.75 & -3296.60 & \textbf{--3287.04} & 1.10  \\
& & Uneven & -3507.93 & -3496.16 & -3493.34 & -3492.69 & \textbf{--3490.43} & -3496.

In [ ]:
from env import InventoryEnv

# Define the LaTeX table structure - same layout as the LVFA theta table above, but
# each cell is that family's initial capacity for facility A/B/C/D/E (deterministic
# given num_warehouses/num_customers/capacity_distribution, see InventoryEnv.__init__
# in env.py) instead of a learned weight.
latex_table = r"""
\begin{table}[H]
    \centering
    \scriptsize
    \caption{Initial facility capacity across all families of instances.}
    \label{tab:facility_initial_capacity_all}
    \begin{tabular}{ccc|S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$A$} & \textbf{$B$} & \textbf{$C$} & \textbf{$D$} & \textbf{$E$} \\
        \midrule
"""

# Loop through the customer, warehouse, and capacity options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Initial capacity per facility - one value per warehouse, so pad with
            # blanks up to the 5-column max (num_warehouses_options tops out at 5)
            MAX_FACILITY_COLS = 5
            env = InventoryEnv(num_warehouses, num_customers, capacity_distribution)
            capacity_values = [str(v) for v in env.warehouses_initial_capacity]
            capacity_cells = capacity_values + [""] * (MAX_FACILITY_COLS - len(capacity_values))
            latex_table += " & ".join(capacity_cells) + " \\\\\n"
    latex_table += "        \\hline\n"
latex_table += "        \\bottomrule\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)



\begin{table}[H]
    \centering
    \scriptsize
    \caption{Initial facility capacity across all families of instances.}
    \label{tab:facility_initial_capacity_all}
    \begin{tabular}{ccc|S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$A$} & \textbf{$B$} & \textbf{$C$} & \textbf{$D$} & \textbf{$E$} \\
        \midrule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & 25 & 25 &  &  &  \\
& & Uneven & 35 & 15 &  &  &  \\
& \multirow{2}{*}{3} & Uniform & 17 & 17 & 16 &  &  \\
& & Uneven & 25 & 15 & 10 &  &  \\
& \multirow{2}{*}{4} & Uniform & 13 & 13 & 12 & 12 &  \\
& & Uneven & 20 & 15 & 10 & 5 &  \\
& \multirow{2}{*}{5} & Uniform & 10 & 10 & 10 & 10 & 10 \\
& & Uneven & 16 & 12 & 10 & 7 & 5 \\
        \hline
        \multirow{8}{*}{100} & \multirow{2}{*}{2} & Uniform & 50 & 50 &  &  &  \\
& & Uneven & 70 & 30 &  &  & 

In [ ]:
# Imitation learning (IL) neural network - architecture and training hyperparameters.
# Values are read from policies/auxiliaries/imitation_learning_model.py (architecture,
# shared with inference) and training/train_imitation_learning.ipynb (training loop),
# not from a results CSV, since these are fixed configuration rather than measured
# outputs.
il_hyperparameters = [
    ("Imitation target (expert policy)", "Exact value function (backward induction)"),
    ("Training instances per family", r"5{,}000 ($|\OrderSet| = 50$ fixed)"),
    ("Input features", r"$2|\FacilitySet|$ (normalized distance and remaining capacity per facility)"),
    ("Architecture", r"Dense($2|\FacilitySet| \to 64$) - ReLU - Dense($64 \to 16$) - ReLU - Dense($16 \to |\FacilitySet|$)"),
    ("Output", r"logits over $|\FacilitySet|$ facilities (predicted action = argmax)"),
    ("Loss function", "Cross-entropy"),
    ("Optimizer", "Adam (default learning rate $= 0.001$)"),
    ("Batch size", "64"),
    ("Epochs", "25"),
    ("Train / validation / test split", r"64\% / 16\% / 20\%"),
    ("Random seed (data split)", "42"),
]

latex_table = r"""
\begin{table}[H]
    \centering
    \scriptsize
    \caption{\gls{il} neural network architecture and training hyperparameters.}
    \label{tab:il_hyperparameters}
    \begin{tabular}{ll}
        \toprule
        \textbf{Parameter} & \textbf{Value} \\
        \midrule
"""

for name, value in il_hyperparameters:
    latex_table += f"        {name} & {value} \\\\\n"

latex_table += r"""        \bottomrule
    \end{tabular}
\end{table}
"""

print(latex_table)



\begin{table}[H]
    \centering
    \scriptsize
    \caption{\gls{il} neural network architecture and training hyperparameters.}
    \label{tab:il_hyperparameters}
    \begin{tabular}{ll}
        \toprule
        \textbf{Parameter} & \textbf{Value} \\
        \midrule
        Imitation target (expert policy) & Exact value function (backward induction) \\
        Training instances per family & 5{,}000 ($|\OrderSet| = 50$ fixed) \\
        Input features & $2|\FacilitySet|$ (normalized distance and remaining capacity per facility) \\
        Architecture & Dense($2|\FacilitySet| \to 64$) - ReLU - Dense($64 \to 16$) - ReLU - Dense($16 \to |\FacilitySet|$) \\
        Output & logits over $|\FacilitySet|$ facilities (predicted action = argmax) \\
        Loss function & Cross-entropy \\
        Optimizer & Adam (default learning rate $= 0.001$) \\
        Batch size & 64 \\
        Epochs & 25 \\
        Train / validation / test split & 64\% / 16\% / 20\% \\
        Random seed (data spl

In [3]:
# =====================================================================================
# TABLE: policy training time (in minutes), per family of instances
# -------------------------------------------------------------------------------------
# Provide a vector of policy names; for each one this reads
# training/{policy}_training/{policy}_training_times.csv (columns: num_warehouses,
# num_customers, capacity_distribution, training_time) and emits a LaTeX table with one
# row per family (same ccc + multirow layout as the other per-family tables above) and
# one column per policy. Unlike the runtime-comparison table above, no
# Average/Min/Max summary rows are added here - just the raw per-family training time.
# A policy missing its CSV, or missing a specific family within it, gets a dash ('--').
# =====================================================================================
import os
import pandas as pd


def generate_training_time_table(
    policies,
    num_warehouses_options=(2, 3, 4, 5),
    num_customers_options=(50, 100, 200, 400),
    capacity_distribution_options=('uniform', 'uneven'),
    policies_dict=None,
    caption=None,
    label='tab:policy_training_time',
):
    """Build a LaTeX table with the training time (in minutes) of each policy in
    `policies`, for each family of instances."""
    default_labels = {
        'myopic': 'MYO', 'imitation_learning': 'IL', 'genetic_programming': 'GP',
        'linear_value_function_approximation': 'LVFA', 'deep_q_networks': 'DQN',
        'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL',
        'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE',
        'parameterized_lookahead_approximation': 'PLA',
        'proximal_policy_optimization': 'PPO', 'exact_value_function': 'OPT',
    }
    labels = {**default_labels, **(policies_dict or {})}

    # Load each policy's training times into {(w, c, d): training_time}
    times_by_policy = {}
    for policy in policies:
        csv_path = f'training/{policy}_training/{policy}_training_times.csv'
        if not os.path.exists(csv_path):
            print(f"[warning] no training-time CSV for '{policy}' at {csv_path}; "
                  f"filling that column with '--'")
            times_by_policy[policy] = {}
            continue
        df = pd.read_csv(csv_path)
        times_by_policy[policy] = {
            (int(r.num_warehouses), int(r.num_customers), r.capacity_distribution): float(r.training_time)
            for r in df.itertuples()
        }

    # Column width (integer digits) per policy, from its own observed max value
    col_format = {}
    for policy in policies:
        values = list(times_by_policy[policy].values())
        max_int_digits = len(str(int(max(values)))) if values else 1
        col_format[policy] = max(max_int_digits, 1)

    header_cols = " & ".join(f"\\textbf{{\\gls{{{labels.get(p, p).lower()}}}}}" for p in policies)
    col_spec = "ccc|" + "".join(f"S[table-format={col_format[p]}.2]" for p in policies)

    if caption is None:
        caption = ("Training time (in minutes) of each policy across all families of "
                   "instances.")

    latex_table = "\\begin{table}[H]\n"
    latex_table += "    \\scriptsize\n    \\centering\n"
    latex_table += f"    \\caption{{{caption}}}\n"
    latex_table += f"    \\label{{{label}}}\n"
    latex_table += f"    \\begin{{tabular}}{{{col_spec}}}\n"
    latex_table += "    \\toprule\n"
    latex_table += (f"    \\textbf{{$|\\OrderSet|$}} & \\textbf{{$|\\FacilitySet|$}} & "
                     f"\\textbf{{Capacity}} & {header_cols} \\\\\n")
    latex_table += "    \\midrule\n"

    rows_per_family = len(num_warehouses_options) * len(capacity_distribution_options)
    for f_idx, num_customers in enumerate(num_customers_options):
        latex_table += f"        \\multirow{{{rows_per_family}}}{{*}}{{{num_customers}}} "
        for num_warehouses in num_warehouses_options:
            for i, capacity_distribution in enumerate(capacity_distribution_options):
                if i == 0:
                    latex_table += f"& \\multirow{{{len(capacity_distribution_options)}}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
                else:
                    latex_table += f"& & {capacity_distribution.capitalize()} & "

                cells = []
                for policy in policies:
                    value = times_by_policy[policy].get((num_warehouses, num_customers, capacity_distribution))
                    cells.append(f"{value:.2f}" if value is not None else "")
                latex_table += " & ".join(cells) + r" \\" + "\n"
        if f_idx < len(num_customers_options) - 1:
            latex_table += "        \\hline\n"

    latex_table += "        \\bottomrule\n"
    latex_table += "    \\end{tabular}\n\\end{table}\n"
    return latex_table


# Provide the policies to include (must have a training/{policy}_training/{policy}_training_times.csv)
training_time_policies = ['genetic_programming', 'exact_value_function']

print(generate_training_time_table(training_time_policies))


[warning] no training-time CSV for 'genetic_programming' at training/genetic_programming_training/genetic_programming_training_times.csv; filling that column with '--'
\begin{table}[H]
    \scriptsize
    \centering
    \caption{Training time (in minutes) of each policy across all families of instances.}
    \label{tab:policy_training_time}
    \begin{tabular}{ccc|S[table-format=1.2]S[table-format=2.2]}
    \toprule
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{gp}} & \textbf{\gls{opt}} \\
    \midrule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform &  & 0.08 \\
& & Uneven &  & 0.07 \\
& \multirow{2}{*}{3} & Uniform &  & 0.86 \\
& & Uneven &  & 0.83 \\
& \multirow{2}{*}{4} & Uniform &  & 7.42 \\
& & Uneven &  & 4.85 \\
& \multirow{2}{*}{5} & Uniform &  & 43.76 \\
& & Uneven &  & 31.02 \\
        \hline
        \multirow{8}{*}{100} & \multirow{2}{*}{2} & Uniform &  &  \\
& & Uneven &  &  \\
& \multirow{2}{*}{3} & Uniform &  &  \\
& 